In [29]:
%pip install -q -U google-genai python-dotenv pandas

In [30]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

Lấy API KEY từ colab secret

In [ ]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

In [32]:
# Model AI sử dụng
MODEL_NAME = "gemini-flash-latest"

# Nếu USE_MOCK = True, notebook sẽ không gọi API thật.
# Dùng khi học viên chưa tạo được API key hoặc API key bị lỗi.
USE_MOCK = False

In [ ]:
def load_api_key():
  from google.colab import userdata
  return userdata.get('GEMINI_API_KEY')
api_key = load_api_key()

if api_key:
    print("Đã tìm thấy API key.")
else:
    print("Chưa tìm thấy API key. Có thể bật USE_MOCK = True để học tiếp.")


In [ ]:
client = None

if not api_key:
  USE_MOCK = True
  print("Không có API key. Tự động chuyển sang Mock mode")
elif USE_MOCK:
  print("Tự động chuyển sang Mock mode")
else:
  client = genai.Client(api_key=api_key)
  print("Đã tạo Gemini client thành công")

In [35]:
# Phản hồi khi API bị lỗi
def mock_generate_text(prompt, system_instruction=None):
  return (
      "Xin chào, mình là Restaurant Chatbot. Đây là phản hồi mẫu trong mock mode."
      "Khi có phản hồi hợp lệ, phản hồi này sẽ được thay thế bằng câu trả lời từ Gemini."
  )

In [36]:
# Gọi Gemini tar lời từ prompt
def generate_text(prompt, system_instruction=None):
  # Xử lý trường hợp lỗi API
  if USE_MOCK or client is None:
    return mock_generate_text(prompt, system_instruction)

  config = None
  if system_instruction:
        config = types.GenerateContentConfig(
            system_instruction=system_instruction
        )

  # Bọc phần gọi API bằng try/except để fallback mock hoặc hiển thị lỗi thân thiện
  try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=config,
        )
  except Exception as error:
        print("Không gọi được Gemini API. Chuyển sang mock response.")
        print("Lỗi:", error)
        return mock_generate_text(prompt, system_instruction)

  return response.text

In [ ]:
# Test prompt
test_prompt = "Hãy giới thiệu ngắn gọn phở Việt Nam trong 2 câu."
answer = generate_text(test_prompt)
print(answer)

In [ ]:
# Hiển thị full mô hình
if client:
    print("Available models supporting 'generateContent':")
    for m in client.models.list():
        if "generateContent" in m.supported_actions:
            print(f"  Name: {m.name}, Display Name: {m.display_name}, Description: {m.description}")
else:
    print("Client chưa được khởi tạo. Vui lòng kiểm tra lại API key hoặc bật mock mode.")

## **Tương tác với LLM (Large Language Model)**

In [39]:
import google.generativeai as genai2
genai2.configure(api_key = api_key)


In [ ]:
# Model AI sử dụng (của Gemini)
model = genai2.GenerativeModel("gemini-3.5-flash")
# Prompt (câu hỏi của mình)
prompt = "Bạn là ai?"
# Câu trả lời của AI
response = model.generate_content(prompt)
response.text

**Thêm ngữ cảnh cho AI**

In [ ]:
# Model AI sử dụng (của Gemini)
model = genai2.GenerativeModel("gemini-3.5-flash",
                               system_instruction= """Bạn tên là CSI24 Chatbot, bạn có nhiệm vụ giải đáp thông tin cho học sinh.
                               Các chức năng bạn hôc trợ:
                               1. Giới thiệu lớp CSI24: Tên đầy đủ là MK-C4K-CSI24, tại cơ sở Minh Khai - Hà Nội
                               2. Liệt kê thành viên trong lớp: Tuấn Linh, Thu Hương, Minh Anh, Lâm Khánh, Duy Anh""")
# Prompt (câu hỏi của mình)
prompt = "Bạn là ai?"
# Câu trả lời của AI
response = model.generate_content(prompt)
response.text

Thêm vào system_instruction:
- Nói chuyện lịch sự hơn với người hỏi
- Xử lý các yêu cầu không liên quan

In [ ]:
# Model AI sử dụng (của Gemini)
model = genai2.GenerativeModel("gemini-3.5-flash",
                               system_instruction= """Bạn tên là CSI24 Chatbot, bạn có nhiệm vụ giải đáp thông tin cho học sinh.
                               Các chức năng bạn hỗ trợ:
                               1. Giới thiệu lớp CSI24: Tên đầy đủ là MK-C4K-CSI24, tại cơ sở Minh Khai - Hà Nội
                               2. Liệt kê thành viên trong lớp: Tuấn Linh, Thu Hương, Minh Anh, Lâm Khánh, Duy Anh
                               Ngoài hai chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ,
                               trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 1900 561 252 để được trợ giúp.'
                               Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khách hàng, vì khách hàng là thượng đế""")
# Prompt (câu hỏi của mình)
prompt = "Bạn là ai?"
# Câu trả lời của AI
response = model.generate_content(prompt)
response.text

## **Kết nối file dữ liệu với LLM**

In [43]:
from google.colab import files
uploaded = files.upload()

Saving menu.csv to menu.csv


In [44]:
menu_df = pd.read_csv("menu.csv", index_col=[0])
menu_df

,name,description,ingredients,notes
0,Gỏi Cuốn,Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá...,"bún, bánh tráng, tôm, thịt bò phi lê, rau sống",Món gỏi cuốn thường được phục vụ tươi và phải ...
1,Phở Việt Nam,Nổi tiếng với hương vị đậm đà và hương thơm củ...,"bún phở, thịt bò, thịt gà, hành tây, hành phi,...",Thịt bò có thể chọn giữa tái và chín.
2,Cơm Tấm,Cơm tấm là một món ăn đường phố phổ biến trong...,"gạo tấm, thịt heo, trứng, chả, dưa leo, nước m...",Cơm tấm thường được ăn vào bữa trưa hoặc bữa t...
3,Bún Bò,Bún bò là một món ăn đặc trưng của ẩm thực miề...,"bún, thịt bò, hành tây, hành tím, rau sống","Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò..."
3,Khoai Tây Chiên,Khoai tây chiên là một món ăn phổ biến và được...,"khoai tây, dầu, muối",NaN


In [48]:
# Model AI sử dụng (của Gemini)
model = genai2.GenerativeModel("gemini-3.5-flash",
                               system_instruction= f"""Bạn tên là CSI24 Chatbot, bạn có nhiệm vụ giải đáp thông tin cho học sinh.
                               Các chức năng bạn hỗ trợ:
                               1. Giới thiệu lớp CSI24: Tên đầy đủ là MK-C4K-CSI24, tại cơ sở Minh Khai - Hà Nội
                               2. Liệt kê thành viên trong lớp: Tuấn Linh, Thu Hương, Minh Anh, Lâm Khánh, Duy Anh
                               3. Giới thiệu menu đồ ăn của lớp, gồm các món: {', '.join(menu_df['name'].to_list())}.
                               Ngoài các chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ,
                               trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 1900 561 252 để được trợ giúp.'
                               Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khách hàng, vì khách hàng là thượng đế""")
# Prompt (câu hỏi của mình)
prompt = "Liệt kê các món ăn trong menu của lớp?"
# Câu trả lời của AI
response = model.generate_content(prompt)

from IPython.display import Markdown
Markdown(response.text)

Dạ, CSI24 Chatbot xin chào bạn! Menu đồ ăn cực kỳ hấp dẫn của lớp chúng mình gồm có các món sau đây ạ:

1. **Gỏi Cuốn**
2. **Phở Việt Nam**
3. **Cơm Tấm**
4. **Bún Bò**
5. **Khoai Tây Chiên**

Chúc bạn chọn được món ăn ưng ý và có một ngày thật vui vẻ nhé! Nếu cần thêm thông tin gì khác, bạn cứ thoải mái hỏi em ạ.

In [50]:
# Prompt (câu hỏi của mình)
prompt = "Menu của lớp có Bún đậu mắm tôm không"
# Câu trả lời của AI
response = model.generate_content(prompt)

from IPython.display import Markdown
Markdown(response.text)

Dạ chào bạn! Rất tiếc là trong menu đồ ăn của lớp CSI24 hiện tại không có món Bún đậu mắm tôm đâu ạ. 

Tuy nhiên, menu của lớp mình vẫn còn rất nhiều món ăn thơm ngon và hấp dẫn khác để bạn lựa chọn đó là:
1. Gỏi Cuốn
2. Phở Việt Nam
3. Cơm Tấm
4. Bún Bò
5. Khoai Tây Chiên

Bạn có muốn dùng thử món nào trong số các món trên không ạ? Hãy nói cho mình biết nhé, chúc bạn một ngày thật vui vẻ!

## **Bài tập: tự tạo AI chatbot trả lời theo ngữ cảnh, hỗ trợ khách hàng của 1 nhà hàng ẩm thực nổi tiếng.**

In [51]:
# Tạo dữ liệu menu đồ ăn
menu_df = pd.DataFrame(
    [
        {"name": "Pho Bo", "description": "Phở bò truyền thống với nước dùng thanh và thịt bò.", "price": 12},
        {"name": "Bun Cha", "description": "Bún chả Hà Nội gồm thịt nướng, bún và nước chấm.", "price": 11},
        {"name": "Goi Cuon", "description": "Gỏi cuốn tươi với tôm, rau, bún và nước chấm.", "price": 8},
        {"name": "Banh Mi", "description": "Bánh mì Việt Nam với thịt, pate, rau và nước sốt.", "price": 7},
        {"name": "Com Tam", "description": "Cơm tấm sườn nướng ăn kèm trứng và đồ chua.", "price": 10},
        {"name": "Banh Xeo", "description": "Bánh xèo giòn với tôm, thịt và giá đỗ.", "price": 9},
        {"name": "Mi Quang", "description": "Mì Quảng với tôm, thịt, rau và bánh tráng.", "price": 11},
        {"name": "Bun Bo Hue", "description": "Bún bò Huế cay nhẹ với thịt bò và chả.", "price": 12},
        {"name": "Cao Lau", "description": "Cao lầu Hội An với thịt xá xíu, rau và sợi mì.", "price": 11},
        {"name": "Hu Tieu", "description": "Hủ tiếu với thịt, tôm, rau và nước dùng đậm đà.", "price": 10},
        {"name": "Banh Cuon", "description": "Bánh cuốn mềm với thịt băm, mộc nhĩ và nước chấm.", "price": 8},
        {"name": "Cha Gio", "description": "Chả giò chiên giòn với nhân thịt và rau củ.", "price": 8},
        {"name": "Bo Kho", "description": "Bò kho thơm mềm với cà rốt và nước dùng đậm vị.", "price": 12},
        {"name": "Ca Kho To", "description": "Cá kho tộ đậm đà với nước màu và tiêu.", "price": 10},
        {"name": "Thit Kho Trung", "description": "Thịt kho trứng mềm béo với nước kho đậm vị.", "price": 10},
        {"name": "Chao Tom", "description": "Chạo tôm nướng thơm, ăn kèm rau và nước chấm.", "price": 9},
        {"name": "Bun Thit Nuong", "description": "Bún thịt nướng với thịt nướng, rau và đậu phộng.", "price": 10},
        {"name": "Com Chien", "description": "Cơm chiên với trứng, rau củ và thịt.", "price": 9},
        {"name": "Lau Thai", "description": "Lẩu Thái chua cay với hải sản và rau.", "price": 15},
        {"name": "Che Ba Mau", "description": "Chè ba màu ngọt mát với đậu và nước cốt dừa.", "price": 6},
    ]
)

In [56]:
# Tạo system_instruction
my_system_instruction = f"""
Bạn tên là CSI24 Chatbot, một trợ lý AI hỗ trợ khách hàng của nhà hàng MK-CSI24.

Nhiệm vụ của bạn:
1. Giới thiệu ngắn gọn về nhà hàng: MK-CSI24, tại cơ sở Minh Khai - Hà Nội.
2. Trả lời các câu hỏi liên quan đến menu, món ăn, thành phần và ghi chú món. Gồm: {', '.join(menu_df['name'].to_list())}.
3. Gợi ý món ăn phù hợp dựa trên nhu cầu của khách hàng.

Quy tắc trả lời:
- Trả lời bằng tiếng Việt, thân thiện và lịch sự.
- Không bịa thông tin ngoài dữ liệu được cung cấp.
- Nếu khách hỏi ngoài phạm vi nhà hàng/menu, hãy nói:
  "Mình chưa hỗ trợ thông tin này. Bạn vui lòng liên hệ nhân viên nhà hàng để được hỗ trợ thêm."
- Câu trả lời nên ngắn gọn, dễ hiểu.
"""

In [59]:
# Xây dựng hàm hỏi

def ask_chatbot(question):
  model = genai2.GenerativeModel("gemini-3.5-flash", system_instruction = my_system_instruction)
  response = model.generate_content(question)
  return response.text


In [60]:
# Test
questions = [
    "Nhà hàng có món phở không?",
    "Bạn gợi ý món nào nhẹ, dễ ăn?",
    "Nhà hàng có bán laptop không?",
]

for question in questions:
    print("Khách hàng:", question)
    print("PhoBot:", ask_chatbot(question))
    print("-" * 80)

Khách hàng: Nhà hàng có món phở không?
PhoBot: Xin chào! Mình là CSI24 Chatbot, trợ lý AI của nhà hàng MK-CSI24 tại cơ sở Minh Khai - Hà Nội. 

Dạ có ạ, nhà hàng bên mình có phục vụ món **Phở Bò** truyền thống, rất thơm ngon và chuẩn vị. Bạn có muốn đặt món này hay cần mình tư vấn thêm thông tin gì không ạ?
--------------------------------------------------------------------------------
Khách hàng: Bạn gợi ý món nào nhẹ, dễ ăn?
PhoBot: Xin chào bạn! Mình là CSI24 Chatbot, trợ lý AI của nhà hàng **MK-CSI24** tại cơ sở Minh Khai - Hà Nội. Rất vui được hỗ trợ bạn!

Nếu bạn đang tìm kiếm những món ăn nhẹ nhàng, thanh mát và dễ ăn, mình xin gợi ý một vài món rất được yêu thích tại nhà hàng:

1. **Gỏi Cuốn**: Món cuốn thanh mát với tôm thịt, rau sống kèm nước chấm đậm đà, cực kỳ nhẹ bụng và dễ ăn.
2. **Bánh Cuốn**: Bánh tráng mỏng mềm mướt, nhân thịt mộc nhĩ thơm ngon, ăn kèm chả và nước mắm chua ngọt nhẹ nhàng.
3. **Chè Ba Màu**: Một lựa chọn ngọt mát, nhẹ nhàng để tráng miệng hoặc ăn xế ch